In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        
        # Filter out files with .txt extension
        self.image_names = [f for f in os.listdir(image_dir) if not f.endswith('.txt')]

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("RGB")  # or "L" for grayscale
        mask = Image.open(mask_path).convert("RGB")    # Convert mask to single channel

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        return image, mask

# Define transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),       # Resize images and masks to 256x256
    transforms.ToTensor(),               # Convert images and masks to PyTorch tensors
    transforms.Normalize([0.5], [0.5])   # Normalize images (mean, std), adjust for RGB if necessary
])

# Create dataset
train_dataset = SegmentationDataset(image_dir="test_practice/test_practice", mask_dir="labelled_images", transform=transform)

# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Example iteration

    # print(images)  # Should print torch.Size([8, 3, 256, 256]) and torch.Size([8, 1, 256, 256])
    

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet, self).__init__()

        # Encoder
        self.enc1 = self.conv_block(in_channels, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)

        # Bottleneck
        self.bottleneck = self.conv_block(512, 1024)

        # Decoder
        self.upconv4 = self.upconv_block(1024, 512)
        self.upconv3 = self.upconv_block(512, 256)
        self.upconv2 = self.upconv_block(256, 128)
        self.upconv1 = self.upconv_block(128, 64)

        # Final Convolution
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        ).cuda()

    def upconv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU(inplace=True)
        ).cuda()

    def forward(self, x):
    # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(F.max_pool2d(enc1, 2))
        enc3 = self.enc3(F.max_pool2d(enc2, 2))
        enc4 = self.enc4(F.max_pool2d(enc3, 2))

        # Bottleneck
        bottleneck = self.bottleneck(F.max_pool2d(enc4, 2))

        # Decoder
        dec4 = self.upconv4(bottleneck)
        
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.conv_block(dec4.shape[1], 512)(dec4)
        dec3 = self.upconv3(dec4)
        
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.conv_block(dec3.shape[1], 256)(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.conv_block(dec2.shape[1], 128)(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.conv_block(dec1.shape[1], 64)(dec1)

        # Final Convolution
        return self.final_conv(dec1)



In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader

# Define hyperparameters
num_epochs = 20
learning_rate = 0.001
batch_size = 8

# Assuming you have a dataset and DataLoader defined
# Example:
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Model, Loss Function, and Optimizer
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        # Apply sigmoid to get probabilities
        logits = torch.sigmoid(logits)
        
        # Flatten the tensors to calculate intersection and union
        logits = logits.view(-1)
        targets = targets.view(-1)
        
        intersection = (logits * targets).sum()
        dice = (2. * intersection + self.smooth) / (logits.sum() + targets.sum() + self.smooth)
        
        return 1 - dice  # Since we want to minimize the loss

model = UNet(in_channels=3, out_channels=1).cuda()
for name, param in model.named_parameters():
    print(f"Parameter {name} is on device: {param.device}")
criterion = DiceLoss()  # For multi-class segmentation
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop
for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0

    for images, masks in train_loader:
        images = images.cuda()
        masks = masks.cuda()

        # Forward pass
        outputs = model(images)
        print(masks.shape, outputs.shape)
        loss = criterion(outputs, masks.long())  # Ensure masks are in long format for CrossEntropyLoss

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

    # Validation step (optional)
    # model.eval()  # Set model to evaluation mode
    # val_loss = 0.0
    # with torch.no_grad():
    #     for val_images, val_masks in val_loader:
    #         val_images = val_images.to('cuda' if torch.cuda.is_available() else 'cpu')
    #         val_masks = val_masks.to('cuda' if torch.cuda.is_available() else 'cpu')
    #         val_outputs = model(val_images)
    #         loss = criterion(val_outputs, val_masks.long())
    #         val_loss += loss.item()

    # print(f"Validation Loss: {val_loss/len(val_loader):.4f}")

# Save the model checkpoint
torch.save(model.state_dict(), 'unet_model.pth')
